In [1]:
import pandas as pd

In [2]:
# Connecting to postgreSQL
import psycopg


conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="banking_practice",
    user="postgres",
    password="manoj"
)


cursor = conn.cursor()


print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [3]:
# Show every ACTIVE account together with the owning customer's full name and email. Only Active accounts should appear.

cursor.execute("""
    select a.account_id,
        a.account_type,
        c.first_name ||' '|| c.last_name as full_name,
        c.email
    from customers as c join accounts as a 
        on a.customer_id = c.customer_id
    where a.status = 'Active'
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(100001, 'Checking', 'Krishna Rai', 'krishna.rai1@mailbank.com')
(100002, 'Fixed Deposit', 'Anita Lama', 'anita.lama2@mailbank.com')
(100004, 'Fixed Deposit', 'Gita Pandey', 'gita.pandey3@mailbank.com')
(100007, 'Checking', 'Indira Gurung', 'indira.gurung5@mailbank.com')
(100008, 'Savings', 'Sabina Bhattarai', 'sabina.bhattarai6@mailbank.com')
(100009, 'Savings', 'Sabina Bhattarai', 'sabina.bhattarai6@mailbank.com')
(100011, 'Recurring Deposit', 'Amit Chaudhary', 'amit.chaudhary8@mailbank.com')
(100012, 'Recurring Deposit', 'Arjun Bhandari', 'arjun.bhandari9@mailbank.com')
(100013, 'Checking', 'Arjun Bhandari', 'arjun.bhandari9@mailbank.com')
(100014, 'Fixed Deposit', 'Sita Maharjan', 'sita.maharjan10@mailbank.com')
(100015, 'Recurring Deposit', 'Kavita KC', 'kavita.kc11@mailbank.com')
(100017, 'Checking', 'Bidya Pandey', 'bidya.pandey13@mailbank.com')
(100018, 'Checking', 'Pramod Malla', 'pramod.malla14@mailbank.com')
(100019, 'Recurring Deposit', 'Pramod Malla', 'pramod.malla14@mailb

In [4]:
# Find every customer who currently has NO account at all.

cursor.execute("""
    select
        c.customer_id,
        c.first_name ||' '|| c.last_name as full_name
    from customers as c 
    left join accounts as a 
        on a.customer_id = c.customer_id
    where a.account_id is null
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(9999, 'Sabina Bhattarai')


In [5]:
# Find every account whose customer_id does not match any row in the customers table (orphaned accounts).

cursor.execute("""
    select a.account_id,
        a.customer_id,
        a.account_type
    from accounts as a 
    left join customers as c 
        on a.customer_id = c.customer_id
    where c.customer_id is null;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(100280, 99999, 'Savings')


In [ ]:
# Produce one result set of every customer and every account regardless of whether a match exists on either side, and label each row as 'Matched', 'No Account' or 'Missing Customer'.



In [5]:
# For every transaction, show the transaction id, amount, account type, branch and the owning customer's full name - a single query joining three tables.

cursor.execute("""
    select t.transaction_id,
        t.amount,
        a.account_type,
        a.branch,
        c.first_name ||' '|| c.last_name as full_name
    from customers as c join accounts as a
    on c.customer_id = a.customer_id
    join transactions as t 
    on a.account_id = t.account_id
""")

rows = cursor.fetchall()
print(rows)

[(7000011, Decimal('272.45'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000017, Decimal('46086.82'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000012, Decimal('9482.80'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000002, Decimal('19581.27'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000005, Decimal('62000.39'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000010, Decimal('22994.74'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000013, Decimal('51194.54'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000014, Decimal('61654.59'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000016, Decimal('70825.65'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000003, Decimal('57068.55'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000006, Decimal('1830.05'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000018, Decimal('63474.32'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000001, Decimal('53248.58'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000008, Decimal('48054.08'

In [16]:
# Find the total balance held at each branch, ordered from highest to lowest.

cursor.execute("""
    select
        a.branch,
        sum(a.balance)as total_balance
    from accounts as a
    group by branch
    order by total_balance desc
""")

rows = cursor.fetchall()
print(rows)

[('Pokhara City', Decimal('10680073.98')), ('Butwal West', Decimal('10638192.58')), ('Itahari Plaza', Decimal('10574591.36')), ('Biratnagar East', Decimal('8855755.92')), ('Bhaktapur Central', Decimal('8556332.73')), ('Lalitpur', Decimal('8113557.86')), ('Dharan North', Decimal('6782927.72')), ('Kathmandu Main', Decimal('6770720.69'))]


In [23]:
# Find the TOP 5 branches by total balance, counting only Active accounts.

cursor.execute("""
    select
        a.branch,
        sum(a.balance)as total_balance
    from accounts as a
    where a.status = 'Active'
    group by branch
    order by total_balance desc
    limit 5
""")

rows = cursor.fetchall()
print(rows)

[('Bhaktapur Central', Decimal('8429354.28')), ('Biratnagar East', Decimal('8104185.98')), ('Itahari Plaza', Decimal('7743224.00')), ('Butwal West', Decimal('7725054.95')), ('Pokhara City', Decimal('7390898.09'))]


In [30]:
# Find account types where the average balance exceeds 50,000. Round the average to 2 decimal places.

cursor.execute("""
    select
        a.account_type,
        round(avg(a.balance), 2) as average_balance
    from accounts as a
    group by account_type
    having avg(a.balance) > 50000
""")

rows = cursor.fetchall()
print(rows)

[('Checking', Decimal('70698.04')), ('Fixed Deposit', Decimal('410096.22')), ('Savings', Decimal('115148.34')), ('Recurring Deposit', Decimal('356654.44'))]


In [35]:
# Count how many accounts each customer holds, and list only customers who hold more than 1 account.

cursor.execute("""
    select
        c.customer_id,
        c.first_name ||' ' || c.last_name as full_name,
        count(a.account_id) as no_of_accounts
    from customers as c join accounts as a
    on c.customer_id = a.customer_id
    group by c.customer_id, c.first_name, c.last_name
    having count(a.account_id) > 1
""")

rows = cursor.fetchall()
print(rows)

[(58, 'Ganesh KC', 3), (184, 'Sagar Shrestha', 2), (116, 'Rita Bhandari', 2), (70, 'Ravi Chaudhary', 2), (52, 'Priya Shrestha', 3), (162, 'Ashok Lama', 3), (84, 'Indira Joshi', 3), (170, 'Sagar Subedi', 2), (176, 'Milan Pandey', 3), (92, 'Poonam Regmi', 2), (101, 'Bina Pandey', 2), (115, 'Rajesh Chaudhary', 3), (59, 'Rabin Neupane', 3), (65, 'Meera Shrestha', 2), (200, 'Poonam Rana', 3), (73, 'Ram Chaudhary', 3), (161, 'Naveen Neupane', 2), (121, 'Prakash Sharma', 2), (119, 'Champa Shrestha', 2), (9, 'Arjun Bhandari', 2), (196, 'Anjali Shrestha', 2), (15, 'Gita Karki', 2), (79, 'Nabin Adhikari', 2), (26, 'Sita Adhikari', 2), (187, 'Sushila Koirala', 2), (77, 'Nabin Acharya', 2), (30, 'Gita Bhattarai', 3), (21, 'Meera Khadka', 2), (131, 'Mina Tamang', 2), (3, 'Gita Pandey', 3), (17, 'Sagar Khadka', 2), (104, 'Deepak Poudel', 2), (165, 'Anjali Tamang', 3), (179, 'Sunita Chaudhary', 2), (35, 'Kavita Basnet', 3), (174, 'Sarala Magar', 3), (45, 'Gita Gurung', 2), (107, 'Mina Rai', 3), (6, '

In [44]:
# Find the branch and account_type combination that has the single highest total transaction amount.

cursor.execute("""
    select
        a.branch,
        a.account_type,
        sum(t.amount) as total_transaction_amount
    from accounts as a join transactions as t
    on a.account_id = t.account_id
    group by a.branch, a.account_type
    order by total_transaction_amount desc
    limit 1;
""")

rows = cursor.fetchall()
print(rows)

[('Butwal West', 'Recurring Deposit', Decimal('7749589.46'))]


In [ ]:
# Find every customer whose COMBINED account balance is greater than the overall average balance across all accounts.

cursor.execute("""
    select
        c.customer_id,
        c.first_name ||' '|| c.last_name as full_name,
        sum(a.balance) as combined_balance
    from customers as c join accounts as a
    on c.customer_id = a.customer_id
    group by c.customer_id, c.first_name, c.last_name
    having sum(a.balance) > (select avg(balance) from accounts)
""")

rows = cursor.fetchall()
print(rows)

[(8, 'Amit Chaudhary', Decimal('657401.46')), (116, 'Rita Bhandari', Decimal('359182.02')), (87, 'Milan Subedi', Decimal('261428.95')), (51, 'Amit Yadav', Decimal('767708.77')), (146, 'Yogesh Magar', Decimal('710913.82')), (70, 'Ravi Chaudhary', Decimal('1228569.86')), (52, 'Priya Shrestha', Decimal('1023241.87')), (162, 'Ashok Lama', Decimal('435604.59')), (132, 'Santosh Pandey', Decimal('772243.01')), (84, 'Indira Joshi', Decimal('636815.43')), (170, 'Sagar Subedi', Decimal('928250.52')), (192, 'Sanjay Malla', Decimal('381760.51')), (176, 'Milan Pandey', Decimal('1041416.63')), (169, 'Kavita Ghimire', Decimal('513593.76')), (115, 'Rajesh Chaudhary', Decimal('1738508.08')), (60, 'Ram Joshi', Decimal('646380.27')), (97, 'Indira Regmi', Decimal('290240.07')), (108, 'Sagar Maharjan', Decimal('369593.15')), (59, 'Rabin Neupane', Decimal('942183.14')), (124, 'Sunita Lama', Decimal('748602.26')), (200, 'Poonam Rana', Decimal('576165.80')), (73, 'Ram Chaudhary', Decimal('983174.83')), (103, 

In [15]:
# Find accounts whose balance is above the average balance of their own account_type (correlated subquery).

cursor.execute("""
    select a.account_id,
    a.account_type,
    a.balance from accounts as a
    where a.balance > (
        select round(avg(a2.balance),2) from accounts as a2
        where a2.account_type = a.account_type
    );
""")

rows = cursor.fetchall()
print(rows)

[(100001, 'Checking', Decimal('179640.10')), (100002, 'Fixed Deposit', Decimal('757474.71')), (100005, 'Checking', Decimal('136843.80')), (100006, 'Savings', Decimal('219389.39')), (100008, 'Savings', Decimal('252651.41')), (100011, 'Recurring Deposit', Decimal('657401.46')), (100012, 'Recurring Deposit', Decimal('860944.19')), (100016, 'Savings', Decimal('150476.28')), (100018, 'Checking', Decimal('122447.42')), (100019, 'Recurring Deposit', Decimal('390600.78')), (100020, 'Fixed Deposit', Decimal('489768.32')), (100026, 'Fixed Deposit', Decimal('503756.77')), (100028, 'Checking', Decimal('125547.65')), (100032, 'Fixed Deposit', Decimal('489446.26')), (100034, 'Checking', Decimal('151798.60')), (100035, 'Recurring Deposit', Decimal('790202.08')), (100036, 'Recurring Deposit', Decimal('772454.62')), (100041, 'Recurring Deposit', Decimal('707459.36')), (100044, 'Checking', Decimal('74750.72')), (100046, 'Savings', Decimal('266499.00')), (100049, 'Recurring Deposit', Decimal('617890.44')

In [19]:
# Using EXISTS, find every customer who has made at least one 'Withdrawal' transaction.

cursor.execute("""
    select 
        c.customer_id,
        c.first_name ||' '|| c.last_name as full_name
    from customers as c
    where exists (
        select 1
        from accounts as a
        join transactions as t
            on a.account_id = t.account_id
        where a.customer_id = c.customer_id
        and t.txn_type = 'Withdrawal'
    );
""")

rows = cursor.fetchall()
print(rows)

[(1, 'Krishna Rai'), (2, 'Anita Lama'), (3, 'Gita Pandey'), (4, 'Mina Pandey'), (5, 'Indira Gurung'), (6, 'Sabina Bhattarai'), (8, 'Amit Chaudhary'), (9, 'Arjun Bhandari'), (10, 'Sita Maharjan'), (11, 'Kavita KC'), (12, 'Indira Lama'), (13, 'Bidya Pandey'), (14, 'Pramod Malla'), (15, 'Gita Karki'), (17, 'Sagar Khadka'), (18, 'Bina Basnet'), (19, 'Ganesh Sharma'), (20, 'Vikram Subedi'), (21, 'Meera Khadka'), (22, 'Anita Bhattarai'), (23, 'Bidya Acharya'), (24, 'Poonam Magar'), (25, 'Sunita Dahal'), (26, 'Sita Adhikari'), (28, 'Sunita Koirala'), (29, 'Kalpana Ghimire'), (30, 'Gita Bhattarai'), (31, 'Kalpana Koirala'), (32, 'Devi Dahal'), (33, 'Suresh Karki'), (34, 'Poonam Bhattarai'), (35, 'Kavita Basnet'), (36, 'Sanjay Bhandari'), (37, 'Anita Adhikari'), (38, 'Vikram KC'), (39, 'Sita Neupane'), (40, 'Priya Lama'), (41, 'Kamala Karki'), (42, 'Ashok Shrestha'), (43, 'Bikash Acharya'), (45, 'Gita Gurung'), (46, 'Rajesh Adhikari'), (49, 'Sabina Neupane'), (51, 'Amit Yadav'), (52, 'Priya Shr

In [20]:
# Using NOT EXISTS, find every account that has never had a single transaction.

cursor.execute("""
    select 
        a.account_id,
        a.customer_id,
        a.balance,
        a.account_type
    from accounts as a
    where not exists (
        select 1
        from transactions as t
        where a.account_id = t.account_id
    );
""")

rows = cursor.fetchall()
print(rows)

[(100280, 99999, Decimal('15000.00'), 'Savings')]


In [23]:
# Using IN with a subquery, list customers who live in a city that has more than 3 customers.

cursor.execute("""
    select 
        c.customer_id,
        c.first_name ||' ' || c.last_name as full_name,
        c.city
    from customers as c
    where city in (
        select city from customers
        where city is not null
        group by city
        having count(*) > 3
    );
""")

rows = cursor.fetchall()
print(rows)

[(1, 'Krishna Rai', 'Pokhara'), (2, 'Anita Lama', 'Biratnagar'), (3, 'Gita Pandey', 'Biratnagar'), (4, 'Mina Pandey', 'Bhaktapur'), (5, 'Indira Gurung', 'Butwal'), (6, 'Sabina Bhattarai', 'Pokhara'), (7, 'Suresh Tamang', 'Dhangadhi'), (8, 'Amit Chaudhary', 'Janakpur'), (9, 'Arjun Bhandari', 'Janakpur'), (10, 'Sita Maharjan', 'Nepalgunj'), (12, 'Indira Lama', 'Biratnagar'), (13, 'Bidya Pandey', 'Bharatpur'), (14, 'Pramod Malla', 'Damak'), (15, 'Gita Karki', 'Butwal'), (16, 'Sunita Dahal', 'Nepalgunj'), (17, 'Sagar Khadka', 'Pokhara'), (18, 'Bina Basnet', 'Bhaktapur'), (19, 'Ganesh Sharma', 'Janakpur'), (20, 'Vikram Subedi', 'Nepalgunj'), (21, 'Meera Khadka', 'Dhangadhi'), (22, 'Anita Bhattarai', 'Butwal'), (23, 'Bidya Acharya', 'Dhangadhi'), (24, 'Poonam Magar', 'Lalitpur'), (25, 'Sunita Dahal', 'Kathmandu'), (26, 'Sita Adhikari', 'Hetauda'), (27, 'Gita KC', 'Itahari'), (28, 'Sunita Koirala', 'Dhangadhi'), (29, 'Kalpana Ghimire', 'Kathmandu'), (30, 'Gita Bhattarai', 'Kathmandu'), (31, '

In [52]:
# Using a subquery in the FROM clause (inline view), compute the number of accounts and average balance per branch, then keep only branches with more than 5 accounts.

cursor.execute("""
    select 
        branch,
        account_count,
        avg_balance
    from (
        select 
            branch,
            count(*) as account_count,
            round(avg(balance), 2) as avg_balance
        from accounts
        group by branch
    ) as branch_summary
    where account_count > 5;
""")

rows = cursor.fetchall()
print(rows)

[('Biratnagar East', 36, Decimal('245993.22')), ('Lalitpur', 27, Decimal('300502.14')), ('Pokhara City', 41, Decimal('260489.61')), ('Itahari Plaza', 39, Decimal('271143.37')), ('Butwal West', 35, Decimal('303948.36')), ('Kathmandu Main', 36, Decimal('188075.57')), ('Bhaktapur Central', 38, Decimal('225166.65')), ('Dharan North', 28, Decimal('242247.42'))]


In [53]:
# Combine the customer ids that hold a Savings account with the customer ids that hold a Checking account into ONE de-duplicated list, using UNION.

cursor.execute("""
    select 
        customer_id
    from accounts
    where account_type = 'Savings'

    union

    select 
        customer_id
    from accounts
    where account_type = 'Checking';
""")

rows = cursor.fetchall()
print(rows)

[(184,), (116,), (71,), (68,), (52,), (162,), (84,), (170,), (101,), (69,), (180,), (114,), (115,), (112,), (156,), (59,), (197,), (65,), (98,), (173,), (200,), (73,), (44,), (189,), (161,), (88,), (188,), (119,), (43,), (147,), (9,), (196,), (15,), (79,), (48,), (187,), (85,), (57,), (77,), (30,), (21,), (131,), (3,), (28,), (104,), (5,), (165,), (151,), (54,), (4,), (138,), (34,), (90,), (105,), (35,), (45,), (99999,), (174,), (107,), (6,), (134,), (39,), (89,), (36,), (31,), (102,), (14,), (167,), (109,), (13,), (155,), (133,), (111,), (199,), (75,), (128,), (99,), (142,), (46,), (53,), (32,), (183,), (38,), (136,), (150,), (193,), (12,), (137,), (78,), (191,), (25,), (141,), (122,), (186,), (33,), (1,), (106,), (18,), (110,), (178,), (145,), (55,), (129,), (143,), (58,)]


In [54]:
# Produce the same combined Savings/Checking customer list but KEEP duplicates (a customer with both types should appear twice), using UNION ALL.

cursor.execute("""
    select 
        customer_id
    from accounts
    where account_type = 'Savings'

    union all

    select 
        customer_id
    from accounts
    where account_type = 'Checking';
""")

rows = cursor.fetchall()
print(rows)

[(3,), (4,), (6,), (6,), (12,), (18,), (25,), (28,), (31,), (31,), (32,), (34,), (35,), (36,), (39,), (43,), (48,), (55,), (65,), (68,), (69,), (71,), (73,), (77,), (77,), (84,), (84,), (90,), (98,), (99,), (99,), (101,), (102,), (107,), (111,), (115,), (116,), (119,), (131,), (133,), (136,), (137,), (143,), (145,), (150,), (155,), (156,), (167,), (170,), (174,), (183,), (184,), (188,), (189,), (191,), (193,), (199,), (99999,), (1,), (3,), (5,), (9,), (13,), (14,), (15,), (21,), (25,), (30,), (31,), (33,), (38,), (44,), (45,), (46,), (52,), (53,), (54,), (55,), (57,), (58,), (59,), (73,), (75,), (78,), (79,), (85,), (88,), (89,), (104,), (105,), (106,), (107,), (107,), (109,), (109,), (110,), (112,), (114,), (119,), (122,), (128,), (129,), (131,), (134,), (138,), (141,), (142,), (147,), (151,), (161,), (161,), (162,), (162,), (165,), (173,), (178,), (180,), (184,), (186,), (187,), (187,), (196,), (197,), (200,), (200,)]


In [55]:
# Find customer ids that appear in BOTH the Savings list and the Checking list, using INTERSECT.

cursor.execute("""
    select 
        customer_id
    from accounts
    where account_type = 'Savings'

    intersect

    select 
        customer_id
    from accounts
    where account_type = 'Checking';
""")

rows = cursor.fetchall()
print(rows)

[(184,), (119,), (107,), (25,), (31,), (131,), (3,), (55,), (73,)]


In [56]:
# Find customer ids that have a Savings account but do NOT have a Fixed Deposit account, using EXCEPT.

cursor.execute("""
    select 
        customer_id
    from accounts
    where account_type = 'Savings'

    except

    select 
        customer_id
    from accounts
    where account_type = 'Fixed Deposite';
""")

rows = cursor.fetchall()
print(rows)

[(184,), (116,), (99,), (189,), (71,), (68,), (4,), (34,), (188,), (119,), (43,), (32,), (90,), (35,), (99999,), (174,), (183,), (136,), (150,), (107,), (6,), (193,), (84,), (48,), (12,), (170,), (137,), (39,), (191,), (101,), (77,), (69,), (36,), (25,), (31,), (115,), (102,), (167,), (131,), (3,), (28,), (156,), (155,), (133,), (65,), (111,), (18,), (199,), (145,), (55,), (143,), (98,), (73,)]


In [71]:
# Write a CTE that calculates each account's total transaction amount, then use it to list only accounts whose total exceeds 100,000.

cursor.execute("""
    with account_totals as(
        select 
            a.account_id,
            a.account_type,
            a.balance,
            sum(t.amount) as transaction_amount
        from accounts as a join transactions as t
            on a.account_id = t.account_id
        group by a.account_id
    )
    select * from account_totals 
    where transaction_amount > 100000
""")

rows = cursor.fetchall()
print(rows)

[(100186, 'Recurring Deposit', Decimal('485023.71'), Decimal('478532.31')), (100076, 'Savings', Decimal('0.00'), Decimal('111818.73')), (100027, 'Fixed Deposit', Decimal('282995.21'), Decimal('297466.94')), (100258, 'Savings', Decimal('1430.80'), Decimal('362687.83')), (100092, 'Fixed Deposit', Decimal('762147.59'), Decimal('435978.59')), (100220, 'Recurring Deposit', Decimal('298164.11'), Decimal('376127.17')), (100169, 'Recurring Deposit', Decimal('895536.41'), Decimal('240590.38')), (100162, 'Fixed Deposit', Decimal('278653.24'), Decimal('985944.16')), (100004, 'Fixed Deposit', Decimal('84465.72'), Decimal('273502.97')), (100001, 'Checking', Decimal('179640.10'), Decimal('748232.00')), (100274, 'Recurring Deposit', Decimal('341497.07'), Decimal('378471.88')), (100080, 'Recurring Deposit', Decimal('0.00'), Decimal('228850.51')), (100006, 'Savings', Decimal('219389.39'), Decimal('994561.38')), (100054, 'Checking', Decimal('74652.52'), Decimal('801726.79')), (100233, 'Fixed Deposit', D

In [78]:
# Write a CTE to find the single highest-balance account in EACH branch.

cursor.execute("""
    with highest_balance_account as(
        select 
            a.branch,
            max(a.balance) as highest_balance
        from accounts as a 
        group by a.branch
    )
    select 
        a.account_id,
        a.account_type,
        a.branch,
        a.balance
    from accounts as a 
    join highest_balance_account as h
        on a.branch = h.branch
        and a.balance = h.highest_balance 
""")

rows = cursor.fetchall()
print(rows)

[(100012, 'Recurring Deposit', 'Kathmandu Main', Decimal('860944.19')), (100083, 'Recurring Deposit', 'Lalitpur', Decimal('882412.75')), (100151, 'Fixed Deposit', 'Butwal West', Decimal('884143.08')), (100161, 'Fixed Deposit', 'Itahari Plaza', Decimal('874917.08')), (100169, 'Recurring Deposit', 'Dharan North', Decimal('895536.41')), (100172, 'Recurring Deposit', 'Biratnagar East', Decimal('838229.39')), (100202, 'Recurring Deposit', 'Bhaktapur Central', Decimal('835277.30')), (100246, 'Recurring Deposit', 'Pokhara City', Decimal('868569.30'))]


In [3]:
# Chain two CTEs together: the first totals Deposit transactions per account, the second joins that total to accounts and returns accounts whose total deposits exceed their current balance.

cursor.execute("""
   WITH deposit_total AS (
        SELECT
            account_id,
            sum(amount) as total_deposit
        from transactions 
        where txn_type = 'Deposit'
        group by account_id
    ),
    account_deposit as (
        select
            a.account_id,
            a.account_type,
            a.balance,
            d.total_deposit
        from accounts as a 
        join deposit_total as d
            on a.account_id = d.account_id
    )
    select
        account_id,
        account_type,
        balance,
        total_deposit
    from account_deposit
    where total_deposit > balance
""")

rows = cursor.fetchall()
print(rows)

[(100001, 'Checking', Decimal('179640.10'), Decimal('226966.87')), (100007, 'Checking', Decimal('4350.72'), Decimal('169799.90')), (100017, 'Checking', Decimal('19677.46'), Decimal('164433.41')), (100023, 'Recurring Deposit', Decimal('79217.61'), Decimal('93734.21')), (100025, 'Savings', Decimal('104592.30'), Decimal('165393.61')), (100028, 'Checking', Decimal('125547.65'), Decimal('157070.35')), (100029, 'Fixed Deposit', Decimal('76317.99'), Decimal('143433.86')), (100033, 'Savings', Decimal('0.00'), Decimal('26143.72')), (100034, 'Checking', Decimal('151798.60'), Decimal('153924.17')), (100040, 'Checking', Decimal('-3191.86'), Decimal('56575.65')), (100042, 'Recurring Deposit', Decimal('84605.14'), Decimal('87579.72')), (100044, 'Checking', Decimal('74750.72'), Decimal('100807.52')), (100045, 'Savings', Decimal('110313.98'), Decimal('228323.20')), (100047, 'Checking', Decimal('3311.94'), Decimal('184004.86')), (100051, 'Savings', Decimal('51575.47'), Decimal('205659.44')), (100052, '

In [5]:
# Create a VIEW named active_accounts_view exposing only Active accounts along with the owning customer's full name.

cursor.execute("""
    create or replace view active_accounts_view as
        select 
            a.account_id,
            c.customer_id,
            a.account_type,
            c.first_name ||' '|| c.last_name as full_name,
            a.status
        from accounts as a 
        join customers as c
            on a.customer_id = c.customer_id
        where a.status = 'Active'

""")

cursor.execute("""
    select * from active_accounts_view
""")

rows = cursor.fetchall()
print(rows)

[(100001, 1, 'Checking', 'Krishna Rai', 'Active'), (100002, 2, 'Fixed Deposit', 'Anita Lama', 'Active'), (100004, 3, 'Fixed Deposit', 'Gita Pandey', 'Active'), (100007, 5, 'Checking', 'Indira Gurung', 'Active'), (100008, 6, 'Savings', 'Sabina Bhattarai', 'Active'), (100009, 6, 'Savings', 'Sabina Bhattarai', 'Active'), (100011, 8, 'Recurring Deposit', 'Amit Chaudhary', 'Active'), (100012, 9, 'Recurring Deposit', 'Arjun Bhandari', 'Active'), (100013, 9, 'Checking', 'Arjun Bhandari', 'Active'), (100014, 10, 'Fixed Deposit', 'Sita Maharjan', 'Active'), (100015, 11, 'Recurring Deposit', 'Kavita KC', 'Active'), (100017, 13, 'Checking', 'Bidya Pandey', 'Active'), (100018, 14, 'Checking', 'Pramod Malla', 'Active'), (100019, 14, 'Recurring Deposit', 'Pramod Malla', 'Active'), (100020, 15, 'Fixed Deposit', 'Gita Karki', 'Active'), (100021, 15, 'Checking', 'Gita Karki', 'Active'), (100023, 17, 'Recurring Deposit', 'Sagar Khadka', 'Active'), (100024, 17, 'Fixed Deposit', 'Sagar Khadka', 'Active'),

In [13]:
# Create a MATERIALIZED VIEW named branch_balance_summary that pre-aggregates total balance and account count per branch, and write the command to refresh it CONCURRENTLY.

cursor.execute("""
    create materialized view branch_balance_summary as
        select 
            a.branch,
            sum(a.balance) as total_balance,
            count(*) as account_count
        from accounts as a 
        group by a.branch

""")

cursor.execute("""
    create unique index branch_balance_summary_uq
        on branch_balance_summary(branch)
""")

cursor.execute("""
    refresh materialized view concurrently branch_balance_summary
""")

cursor.execute("""
    SELECT *
    FROM branch_balance_summary
""")

rows = cursor.fetchall()
print(rows)

[('Biratnagar East', Decimal('8855755.92'), 36), ('Lalitpur', Decimal('8113557.86'), 27), ('Pokhara City', Decimal('10680073.98'), 41), ('Itahari Plaza', Decimal('10574591.36'), 39), ('Butwal West', Decimal('10638192.58'), 35), ('Kathmandu Main', Decimal('6770720.69'), 36), ('Bhaktapur Central', Decimal('8556332.73'), 38), ('Dharan North', Decimal('6782927.72'), 28)]


In [ ]:
# Using ROW_NUMBER(), return only the MOST RECENT transaction for every account.

cursor.execute("""
    select 
        a.account_id,
        t.transaction_id,
        
    row_number() over(

    )
""")

In [35]:
# Using RANK(), rank customers by their total account balance so that tied balances share the same rank (with a gap afterward).

In [36]:
# Using DENSE_RANK(), rank branches by total transaction amount with NO gaps in the ranking numbers.

In [37]:
# Using LAG(), show each transaction next to the amount of the PREVIOUS transaction on the same account, ordered by date.

In [38]:
# Using LEAD(), show each transaction next to the amount of the NEXT transaction on the same account, and calculate the difference between them.

In [39]:
# Using a running-total window function, show every account's transactions in date order with a cumulative (running) amount.

In [40]:
# Find duplicate customer records - customers who share the exact same first_name, last_name and dob.

In [41]:
# Write ONE query that finds customers missing a city or email (NULL or blank), and a SECOND query that finds orphaned accounts (customer_id with no matching customer row).

In [42]:
# Using CASE WHEN, bucket every Active account into 'Low' (< 10,000), 'Medium' (10,000-100,000) or 'High' (> 100,000), then count accounts in each bucket.

In [43]:
# Write a SAFE transaction block that deducts a 500 maintenance fee from every account with balance > 200,000, inserts a matching 'Fee' row into transactions for each of those accounts, and can be rolled back if anything fails. Always filter UPDATE/DELETE with WHERE.

In [44]:
# Using NTILE(4), split customers into 4 equal-sized income quartiles ordered by annual_income, then count how many customers fall in each quartile.

In [45]:
# Find every customer with a credit_score below 500 who still holds at least one account with a balance above 200,000

In [46]:
# List every FLAGGED transaction (is_flagged = true) together with the owning customer's name, the branch and the channel used, ordered by amount descending.

In [47]:
# Find every customer whose kyc_status is 'Expired' but who still has at least one 'Active' account (a compliance risk).

In [48]:
# Find joint accounts (is_joint_account = true) whose balance is above the AVERAGE balance of all accounts in their own branch (correlated subquery).